In [ ]:
#parallel workflow without LLM
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [6]:
class BatsmanState(TypedDict):
    runs: int
    balls:int
    fours:int
    sixes:int

    sr:float
    bpb:float
    boundary_percent:float
    summary=str

In [ ]:
#Remember that when we are sharing in parallel workflow, we will share only the relevent data to them , so when we are returning it so we will not return whole state, we return a dictionary with relevent changes

In [36]:
def calculate_sr(state:BatsmanState)->BatsmanState:
    sr = (state['runs']/state['balls'])*100
    state['sr']=sr

    #return state
    return {'sr':sr}

In [37]:
def calculate_bpb(state : BatsmanState)->BatsmanState:
    bpb= state['balls']/(state['fours']+state['sixes'])

    state['bpb']=bpb

    return {'bpb':bpb}

In [38]:
def calculate_boundary_percent(state: BatsmanState)->BatsmanState:
    boundary_percent=(((state['fours']*4)+(state['sixes']*6))/state['runs'])*100

    state['boundary_percent']=boundary_percent

    return {'boundary_percent':boundary_percent}


In [39]:
def summary(state: BatsmanState):
    summary=f"""
    Strike Rate-{state['sr']}\n 
    Balls Per Boundary-{state['bpb']}\n
    Boundary Percent - {state['boundary_percent']}
    """

    state['summary']=summary

    return {"summary":summary}


In [40]:
#graph

graph= StateGraph(BatsmanState)

#Node
graph.add_node('calculate_sr', calculate_sr)
graph.add_node('calculate_bpb', calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node("summary",summary)

#edges
graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundary_percent')

graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundary_percent','summary')

graph.add_edge('summary',END)

workflow=graph.compile()

In [41]:

intial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}
workflow.invoke(intial_state)

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundary_percent': 48.0}